In [44]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import json

import constants
from model_utils import *
from performance_utils import *

from ast import literal_eval
from pathlib import Path
 
import matplotlib.cm as cm
import ast
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
from pprint import pprint

In [45]:
model_type = constants.RectalCancerStagingData
base_dir = Path.cwd().parent

In [46]:
# Get ORIGINAL DATA
data_path = base_dir / 'data' / constants.RAW_DATA_FILE_NAME
data = pd.read_csv(data_path)
#data = data[['id', 'report_text']]
data.sort_values(by='id', inplace=True, ignore_index=True)


# delete column "posizione" as it is obsolete, renaming "posizione_multiple" in "posizione"
data.drop(columns=['posizione'], inplace=True)
data.rename(columns={'posizione_multiple': 'posizione'}, inplace=True)

sedi_linfonodi = []
for s1, s2 in zip(data.sedi_locoregionali, data.sedi_non_locoregionali):
    sedi_linfonodi.append(str(ast.literal_eval(s1) + ast.literal_eval(s2)))
data['sedi_linfonodi'] = sedi_linfonodi
print(f'Nuova colonna "sedi_linfonodi" creata\n{data.shape = }')

#####################
# Columns aggregation
#####################
# Dettagli infiltrazione organi
infiltrazione_organi_dettagli_new = []
for s in data.infiltrazione_organi_dettagli.fillna(constants.NAN_VALUE):
    dettagli = []
    if s == constants.NAN_VALUE:
        infiltrazione_organi_dettagli_new.append(str(dettagli))
    else:
        if s.strip() == '[object Object]':
            infiltrazione_organi_dettagli_new.append(str(dettagli))
            continue 
        d = ast.literal_eval(s)
        if constants.InfiltrazioneOrganiDettagli.PAVIMENTO_PELVICO.value in d:
            dettagli.append(constants.InfiltrazioneOrganiDettagli.PAVIMENTO_PELVICO.value)
        if ('altro' in d) or ('utero' in d) or ('sacro' in d):
            dettagli.append(constants.InfiltrazioneOrganiDettagli.ALTRO.value)
        infiltrazione_organi_dettagli_new.append(str(dettagli))
data.loc[:, 'infiltrazione_organi_dettagli'] = infiltrazione_organi_dettagli_new


data.set_index('id', inplace=True)
print(f'{data.shape = }')

Nuova colonna "sedi_linfonodi" creata
data.shape = (343, 45)
data.shape = (343, 44)


In [47]:

mistral = load_results_data("new_results_mistral-large-3.jsonl", base_dir/'data'/'inference', model_type)
gpt = load_results_data("new_results_gpt-4.1.jsonl", base_dir/'data'/'inference', model_type)
opus = load_results_data("new_results_opus-4.6.jsonl", base_dir/'data'/'inference', model_type)

assert len(mistral) == len(gpt) == len(opus)


mistral = {
    ex['id']: {
        'actual':       ex['actual'],
        'prediction':   ex['prediction'],
        'split':        ex['split']
    }
    for ex in mistral
}
gpt = {
    ex['id']: {
        'actual':       ex['actual'],
        'prediction':   ex['prediction'],
        'split':        ex['split']
    }
    for ex in gpt
}
opus = {
    ex['id']: {
        'actual':       ex['actual'],
        'prediction':   ex['prediction'],
        'split':        ex['split']
    }
    for ex in opus
}

# Controllo
for id in mistral:
    if mistral[id]['actual'] == gpt[id]['actual'] == opus[id]['actual']:
        continue
    else:
        print(id)

results = {
    id: {
        'actual':   gpt[id]['actual'],
        'gpt':      gpt[id]['prediction'],
        'opus':     opus[id]['prediction'],
        'mistral':  mistral[id]['prediction'],
    }
    for id in gpt
}

reports = {id: data.loc[id]['report_text'] for id in gpt}
profiles = {id: data.loc[id]['profile'] for id in gpt}

In [48]:
def report_model_to_dataframe(report_model: constants.RectalCancerStagingData, name: str) -> pd.DataFrame:
    multilabel_fields = ['posizione', 'infiltrazione_organi_dettagli', 'sedi_linfonodi']    
    model_dict = report_model.model_dump(mode='json')
    
    for f in multilabel_fields:
        label_list = []
        for l, v in model_dict[f].items():
            if v == constants.Flag.SI.value:
                label_list.append(l)
        model_dict[f] = label_list
    s = pd.Series(model_dict)
    s.name = name
    return s.to_frame()

In [49]:
def original_to_df(id: int, original_data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    result = original_data.loc[id, columns].T
    result.name = 'original'
    return result.to_frame()

In [50]:
def highlight_diff(row, color: str = 'red'):
    ref = row.iloc[1]  # prima colonna come riferimento
    styles = ['', '']  # nessuno stile per la colonna di riferimento
    for val in row.iloc[2:]:  # ultime tre colonne
        if val != ref:
            styles.append(f'background-color: {color}')
        else:
            styles.append('')
    return styles

In [51]:
print(reports.keys())

dict_keys([52, 55, 59, 64, 72, 81, 83, 90, 92, 95, 99, 103, 105, 107, 112, 117, 121, 122, 128, 136, 137, 144, 156, 162, 168, 169, 173, 176, 180, 181, 183, 186, 189, 190, 194, 197, 199, 201, 203, 206, 207, 208, 211, 212, 216, 221, 222, 223, 226, 227, 230, 232, 233, 234, 235, 237, 239, 241, 242, 244, 245, 246, 247, 248, 250, 251, 252, 255, 256, 257, 258, 261, 262, 263, 269, 271, 272, 275, 276, 283, 288, 290, 292, 293, 294, 296, 297, 298, 300, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 314, 315, 316, 318, 320, 322, 323, 324, 325, 326, 329, 330, 331, 332, 334, 335, 336, 337, 338, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 376, 377, 379, 380, 381, 382, 383, 384, 386, 387, 389, 392, 393, 394, 396, 397, 398, 400, 401, 402, 403, 407, 409, 410, 411, 412, 413, 415, 416, 418, 419, 420, 421, 422, 51, 57, 61, 66, 73, 75, 77, 79, 85, 98, 104, 110, 116, 119, 125, 

In [52]:
id = 362
original = original_to_df(id, data, list(constants.RectalCancerStagingData.model_fields.keys()))
dfs = [original] + [report_model_to_dataframe(r, n) for n, r in results[id].items()]
df = pd.concat(dfs, axis=1)
print(data.loc[id].interpretazioni)
styled = df.style.apply(highlight_diff, color='red', axis=1)

nan


In [53]:
print(id, profiles[id])
pprint(reports[id])

362 PietroPaoloAzzaro
("RM PELVI L'ESAME STATO ESEGUITO MEDIANTE SEQUENZE FSE,DWI, 3D LAVA DOPO MDC. "
 'IN CORRISPONDENZA DEL RETTO BASSO, PARETE POSTERIORE, DALLA GIUNZIONE '
 'RETTO-ANALE CON ESTENSIONE CRANIALE PER CIRCA 3 CM PRESENTE ISPESSIMENTO '
 'PARIETALE A PLACCA CHE OCCUPA DUE QUARTI DELLA CIRCONFERENZA DEL LUME E SI '
 'ESTENDE CAUDALMENTE AD INTERESSARE IL TERZO SUPERIORE DEL CANALE ANALE, A '
 'SINISTRA, SENZA INTERESSARE IL PIANO INTERSFINTERIALE. LA NEOPLASIA, '
 'MODERATAMENTE IPERINTENSA NELLE SEQUENZE T2 DIPENDENTI, INTERESSA LA PARETE '
 'A TUTTO SPESSORE, SCONFINA IN ALCUNI PUNTI NEL MESORETTO E CONTRAE RAPPORTI '
 "DI CONTIGUIT CON IL MUSCOLO ELEVATORE DELL'ANO SENZA EVIDENTI SEGNI DI "
 'INFILTRAZIONE. NEL MESORETTO DI SINISTRA SONO PRESENTI DUE LINFONODI IL '
 'MAGGIORE DEI QUALI DEL DIAMETRO DI CIRCA 6 MM A LIMITI SFUMATI ED INTENSIT '
 'DI DI SEGNALE DISOMOGENEA SOSPETTO PER LOCALIZZAZIONE DI MALATTIA . VARICI '
 'PELVICHE BILATERALI. UTERO AUMENTATO DI DIMEN

In [54]:
print(id)
display(styled)
print(id, profiles[id])
pprint(reports[id])

362


,original,actual,gpt,opus,mistral
morfologia,solido_polipoide,solido_polipoide,solido_anulare,solido_anulare,solido_anulare
ore_inizio,nan,3,None,3,4
ore_fine,nan,9,None,9,8
spessore_parietale,nan,None,None,None,None
estensione_cranio_caudale,30.000000,30,30,30,30
distanza_oai,nan,0,0,0,0
posizione,['basso'],['basso'],"['basso', 'giunzione']",['basso'],['basso']
riflessione_peritoneale_anteriore,nan,non_descritto,sotto,sotto,sotto
infiltrazione_tessuto_adiposo,si_5mm_plus,si_5mm_plus,si_5mm,si_5mm,si_5mm
infiltrazione_sfinteri,nan,no,no,no,no


362 PietroPaoloAzzaro
("RM PELVI L'ESAME STATO ESEGUITO MEDIANTE SEQUENZE FSE,DWI, 3D LAVA DOPO MDC. "
 'IN CORRISPONDENZA DEL RETTO BASSO, PARETE POSTERIORE, DALLA GIUNZIONE '
 'RETTO-ANALE CON ESTENSIONE CRANIALE PER CIRCA 3 CM PRESENTE ISPESSIMENTO '
 'PARIETALE A PLACCA CHE OCCUPA DUE QUARTI DELLA CIRCONFERENZA DEL LUME E SI '
 'ESTENDE CAUDALMENTE AD INTERESSARE IL TERZO SUPERIORE DEL CANALE ANALE, A '
 'SINISTRA, SENZA INTERESSARE IL PIANO INTERSFINTERIALE. LA NEOPLASIA, '
 'MODERATAMENTE IPERINTENSA NELLE SEQUENZE T2 DIPENDENTI, INTERESSA LA PARETE '
 'A TUTTO SPESSORE, SCONFINA IN ALCUNI PUNTI NEL MESORETTO E CONTRAE RAPPORTI '
 "DI CONTIGUIT CON IL MUSCOLO ELEVATORE DELL'ANO SENZA EVIDENTI SEGNI DI "
 'INFILTRAZIONE. NEL MESORETTO DI SINISTRA SONO PRESENTI DUE LINFONODI IL '
 'MAGGIORE DEI QUALI DEL DIAMETRO DI CIRCA 6 MM A LIMITI SFUMATI ED INTENSIT '
 'DI DI SEGNALE DISOMOGENEA SOSPETTO PER LOCALIZZAZIONE DI MALATTIA . VARICI '
 'PELVICHE BILATERALI. UTERO AUMENTATO DI DIMEN